# match the chl dataset with id tags to the OWT data

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import ticker,colors
import matplotlib.style as style
from matplotlib.ticker import FuncFormatter
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from scipy import stats
import math
import xarray as xr
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import geopandas as gpd
from datetime import datetime,date, timedelta
import datetime as dt 
import cmocean as cmoc
import cmocean.cm as cmo
from erddapy import ERDDAP
import time
import requests
from scipy import stats, odr
import seaborn as sns
from math import radians, sin, cos, sqrt, atan2
import re
import warnings
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
import os
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import ast
import matplotlib.gridspec as gridspec
from statistics import mean


In [15]:
def haversine_distance_vectorized(lat1, lon1, lat2, lon2):
    """
    Calculate the haversine distance between two sets of points in a vectorized way.
    
    Parameters:
        lat1, lon1: Latitude and longitude of the first set of points (arrays).
        lat2, lon2: Latitude and longitude of the second set of points (arrays).
    
    Returns:
        Array of distances in kilometers.
    """
    R = 6371  # Earth radius in kilometers
    
    # Convert latitude and longitude from degrees to radians
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)

    # Calculate differences in coordinates
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    # Apply haversine formula
    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c
    
    return distance

In [22]:
def slice_before_concat(ds):
        """
        Slices the dataset to keep only a specific lat/lon range.
    NOTE: lat and lon have to have same naming conventions as the satellite nc file, this means you might need to change it to latitude and longitude
        """
        # Example: Slice the 'time' dimension to keep only data after a certain date
        ds = ds.sel(lat=slice(80,10),lon=slice(-179,-40))
        return ds

In [31]:
#sat_insit_match
#making the matching data code a function
def sat_insit_match3(sat_xr, insit_df):
    ''' takes a satellite xarray and a in situ dataframe and finds datapoinst with same lat/lon/datetime 
        sat_xr: satellite xarray with variable as same name as matching variable.
        insit_df: in situ dataframe with variable as same name as matching variable
        return: insitu dataframe with additional colum for satellite matches 
    '''
    time_indexer = xr.DataArray(insit_df['datetime'], dims='point')
    lat_indexer = xr.DataArray(insit_df['lat'], dims='point')
    lon_indexer = xr.DataArray(insit_df['lon'], dims='point')

    print("Finding nearest satellite points for all in-situ data at once...")
    matched_sat_points = sat_xr.sel(time=time_indexer, lat=lat_indexer,lon=lon_indexer,method='nearest')

    # --- Step 3: Vectorized Filtering ---
    print("Filtering matches based on spatial and temporal criteria...")
    # Calculate spatial distance for all points vectorially.
    distances_km = haversine_distance_vectorized( insit_df['lat'].values,  insit_df['lon'].values, matched_sat_points['lat'].values,
                                                 matched_sat_points['lon'].values)

    # check if the match is on the exact same day.
    time_match_mask = insit_df['datetime'].dt.date.values == pd.to_datetime(matched_sat_points['time'].values).date
    
    # --- Step 4: Extract 3x3 Grid and Finalize Data ---
    insit_df['chl_occci'] = np.nan
    insit_df['matched_sat_time'] = pd.NaT
    insit_df['matched_spatial_dist_km'] = np.nan

    # Get the indices of only the valid matches.
    valid_indices = np.where(time_match_mask)[0]
    print(f"Found {len(valid_indices)} valid matches out of {len(insit_df)} points. Now extracting 3x3 grids...")
    
    if len(valid_indices) > 0:
        # For performance, get coordinate arrays once before the loop.
        sat_lat_coords = sat_xr['lat'].values
        sat_lon_coords = sat_xr['lon'].values

        # Loop ONLY over the valid matches to extract the 3x3 grid.
        for idx in valid_indices:
            # Get the coordinates of the valid matched satellite point.
            matched_lat = matched_sat_points['lat'].values[idx]
            matched_lon = matched_sat_points['lon'].values[idx]
            matched_time = matched_sat_points['time'].values[idx]

            # Find the integer index using the fast 'get_loc' method.
            lat_idx = sat_xr.indexes['lat'].get_loc(matched_lat)
            lon_idx = sat_xr.indexes['lon'].get_loc(matched_lon)
            time_idx = sat_xr.indexes['time'].get_loc(matched_time)

            # Define the 3x3 window boundaries.
            lat_slice = slice(max(0, lat_idx - 1), min(len(sat_lat_coords), lat_idx + 2))
            lon_slice = slice(max(0, lon_idx - 1), min(len(sat_lon_coords), lon_idx + 2))
            
            # Extract the 3x3 grid using integer-based selection (isel).
            grid_data = sat_xr['chlor_a'].isel(time=time_idx, lat=lat_slice, lon=lon_slice)
            grid_values = grid_data.values.flatten()
            
            # Calculate the mode, ignoring NaN values.
            valid_grid_values = grid_values[~np.isnan(grid_values)]
            grid_mean = np.mean(valid_grid_values) if valid_grid_values.size > 0 else np.nan

            # Store the results in the DataFrame.
            insit_df.loc[idx, 'chl_occci'] = grid_mean
            insit_df.loc[idx, 'matched_sat_time'] = matched_time
            insit_df.loc[idx, 'matched_spatial_dist_km'] = distances_km[idx]

    print('Finished matching process!')
    return insit_df

In [20]:
#def loop that pulls only relevant satellite .nc dates files compared to a dataframe and matches 
def insit_match(insitu_df,dates_year):
    '''
    code that produces dataframe of matching satellite points and insitu points.
    input: 
        insitu_df: dataframe with datetime, lat, lon, and variable
        dates_year: the yearmonthday in format of satellite files
    returns:
        satellite xarray of coastlat us, only that year
        matched chlorophyll + variable dataframe, where var_new is the OWT matched datapoints
    '''
    # pull satellite folder names
    #loop through dates and if the sat data exists and matches date, append to list of file paths.
    
    existing_files= []
    for date in sorted(dates_year):
        file_date = r'J:\OCCCI\V6.0\SOURCE\MAPPED_4KM_DAILY\CHL\ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-'+str(date)+'-fv6.0.nc'
        #file_date = str(sat_xr)+str(date)+'-OCCCI-V6.0-GLOBAL_MAPPED-OWC_WEI.nc'
        if os.path.exists(file_date):
            existing_files.append(file_date)
        else:
            print(f"Warning: File not found for date {date}: {file_date}")
            
            #uncomment below if you're getting errors about corrupted files
    #existing_files= []
    #var = 'water_class'
    #for date in sorted(dates_year):
    #    file_date = str(sat_xr)+str(date)+'-OCCCI-V6.0-GLOBAL_MAPPED-OWC_WEI.nc'
        #files.append(file_date)
    #    if os.path.exists(file_date): #if path exists, try to open it
    #        try:
    #            temp_ds = xr.open_dataset(file_date, chunks={}) # Use chunks={} for Dask if needed
     #           _ = temp_ds[var].isel(time=0, latitude=0, longitude=0).compute()
     #           temp_ds.close()
     #           existing_files.append(file_date)
     #       except Exception as e:
     #           print(f"Skipping corrupted or unreadable file: {file_date}") #. Error: {e}
                #temp_ds.close()
     #   else:
     #        print(f"Warning: File not found for date {date}: {file_date}")
    #concat all satellite .nc files into 1        
    #ds_combined = xr.open_mfdataset(existing_files, combine='by_coords')
    coastal_us_combo= xr.open_mfdataset(existing_files, preprocess=slice_before_concat,combine='by_coords')
    print('satellite refiguring complete')
    test_insitu = insitu_df.copy()
    matched_chl = sat_insit_match3(coastal_us_combo, test_insitu)
    return matched_chl  #coastal_us_combo, for now just return matched dataframe for space


In [2]:
chl_all = pd.read_excel(r"C:\Users\gianna.milton\Documents\Python\Coastal_chl_final\all_chl.xlsx")
chl_all['datetime'] = chl_all['datetime'].apply(pd.to_datetime)

In [46]:
chl_test=chl_all.loc[chl_all['depth']<=10].reset_index(drop=True)  #since OC-CCI is sattellite data and only records 1 depth, reduce to just surface waters 
chl_test = chl_test.tail(500) #subset dataframe to see if it works 
unique_dates=chl_test.copy()
unique_dates['date'] = unique_dates.datetime.dt.date #create column of just date
unique_dates = unique_dates.drop_duplicates(subset=['date'], keep='first').astype(str) #find all unique dates and turn to str
unique_dates['date'] = unique_dates['date'].str.replace('-', '') #take out all - 
dates = unique_dates.date

In [ ]:
%%time
chl_test2 =insit_match(chl_test,dates)

In [44]:
chl_test2 = chl_test2[[ 'datetime', 'lat', 'lon', 'chl', 'chl_a', 'depth', 'experiment','chl_occci', 'matched_sat_time', 'matched_spatial_dist_km']]
chl_test2 = chl_test2.dropna(subset=['chl_occci'])
chl_test2

,datetime,lat,lon,chl,chl_a,depth,experiment,chl_occci,matched_sat_time,matched_spatial_dist_km
5,2000-01-08 07:08:00,32.6780,-117.8770,0.57,NaN,2.0,CALCOFI,0.557138,2000-01-08,2.054913
6,2000-01-08 07:08:00,32.6780,-117.8770,0.61,NaN,10.0,CALCOFI,0.557138,2000-01-08,2.054913
7,2000-01-08 11:19:00,32.5130,-118.2150,0.65,NaN,1.0,CALCOFI,0.478131,2000-01-08,1.588427
8,2000-01-08 11:19:00,32.5130,-118.2150,0.66,NaN,10.0,CALCOFI,0.478131,2000-01-08,1.588427
15,2000-01-09 02:14:00,32.0030,-119.2380,0.53,NaN,2.0,CALCOFI,0.455908,2000-01-09,2.150780
...,...,...,...,...,...,...,...,...,...,...
477,2000-04-29 11:32:00,37.9988,-76.2668,15.61,NaN,0.0,LMER-TIES,1.119700,2000-04-29,2.211556
478,2000-04-29 11:32:00,37.9988,-76.2668,18.55,NaN,0.0,LMER-TIES,1.119700,2000-04-29,2.211556
479,2000-04-29 11:32:00,37.9988,-76.2668,12.18,NaN,0.0,LMER-TIES,1.119700,2000-04-29,2.211556
498,2000-04-30 11:39:00,36.9493,-75.9995,2.56,NaN,0.0,LMER-TIES,4.219806,2000-04-30,2.233150
